In [1]:
import numpy as np

from tron_model import TronBatchModel
from tron_controller import RandomController, GreedySpaceController

In [2]:
# Environment parameters
WIDTH = 64
HEIGHT = 48


## 1. Smoke test the vectorized environment

In [3]:
env = TronBatchModel(width=64, height=48, players=2, envs=1024, keep_owner=False, seed=0)
ctrl = RandomController(seed=0)

obs = env.observe_lite()
actions = ctrl.actions(env)
result = env.step(actions)

print("obs shape:", obs.shape)
print("actions shape:", actions.shape)
print("reward shape:", result.reward.shape)
print("done count:", result.done.sum())

obs shape: (1024, 2, 13)
actions shape: (1024, 2)
reward shape: (1024, 2)
done count: 0


## 2. Fast rollout helper

This evaluates a controller over many parallel games.
Use this for baselines, genetic algorithms, and quick policy comparisons.

In [4]:
def evaluate_controller(controller, *, envs=4096, width=WIDTH, height=HEIGHT, players=2, max_ticks=512, seed=0):
    env = TronBatchModel(
        width=width,
        height=height,
        players=players,
        envs=envs,
        keep_owner=False,
        seed=seed,
    )

    total_reward = np.zeros((envs, players), dtype=np.float32)
    wins = np.zeros((envs, players), dtype=np.float32)
    lengths = np.zeros(envs, dtype=np.int32)

    for _ in range(max_ticks):
        actions = controller.actions(env)
        step = env.step(actions)
        total_reward += step.reward

        active = ~step.done
        lengths[active] += 1

        newly_finished = step.done
        if newly_finished.any():
            alive = step.alive[newly_finished]
            winner_mask = alive.sum(axis=1) == 1
            if winner_mask.any():
                finished_ids = np.flatnonzero(newly_finished)
                winner_ids = finished_ids[winner_mask]
                winners = alive[winner_mask].argmax(axis=1)
                wins[winner_ids, winners] = 1.0

            env.auto_reset_done()

    return {
        "mean_reward_per_player": total_reward.mean(axis=0),
        "win_rate_per_player": wins.mean(axis=0),
        "mean_length": lengths.mean(),
    }


print("Random:", evaluate_controller(RandomController(1), envs=2048))
print("Greedy:", evaluate_controller(GreedySpaceController(), envs=2048))

Random: {'mean_reward_per_player': array([ 0.05908203, -0.24267578], dtype=float32), 'win_rate_per_player': array([0.9995117 , 0.99902344], dtype=float32), 'mean_length': np.float64(500.69580078125)}
Greedy: {'mean_reward_per_player': array([-0.04833984, -0.05126953], dtype=float32), 'win_rate_per_player': array([0.03076172, 0.02929688], dtype=float32), 'mean_length': np.float64(511.89013671875)}


## 3. Minimal NumPy MLP policy

This policy runs independently for each player using `observe_lite()`.
It returns actions with shape `[envs, players]`.

This is useful for genetic algorithms because a genome can simply be a flat NumPy vector of weights.

In [5]:
class MLPPolicy:
    def __init__(self, obs_dim, hidden=32, genome=None, rng=None):
        self.obs_dim = int(obs_dim)
        self.hidden = int(hidden)
        self.rng = np.random.default_rng(rng)

        self.n_params = (
            self.obs_dim * self.hidden + self.hidden +
            self.hidden * 3 + 3
        )

        if genome is None:
            genome = self.rng.normal(0, 0.1, size=self.n_params).astype(np.float32)
        else:
            genome = np.asarray(genome, dtype=np.float32)
        self.genome = genome
        self._unpack_genome(genome)

    def _unpack_genome(self, genome):
        assert genome.shape == (self.n_params,)
        i = 0
        n = self.obs_dim * self.hidden
        self.w1 = genome[i:i+n].reshape(self.obs_dim, self.hidden)
        i += n

        self.b1 = genome[i:i+self.hidden]
        i += self.hidden

        n = self.hidden * 3
        self.w2 = genome[i:i+n].reshape(self.hidden, 3)
        i += n

        self.b2 = genome[i:i+3]

    def set_genome(self, genome):
        genome = np.asarray(genome, dtype=np.float32)
        self.genome = genome
        self._unpack_genome(genome)

    def get_genome(self):
        return self.genome

    def logits(self, observations):
        # obs: [envs, players, obs_dim]
        h = np.tanh(observations @ self.w1 + self.b1)
        return h @ self.w2 + self.b2

    def actions(self, model):
        observations = model.observe_lite()
        logits = self.logits(observations)
        return logits.argmax(axis=-1).astype(np.int8)


probe_env = TronBatchModel(width=WIDTH, height=HEIGHT, players=2, envs=1)
obs_dim = probe_env.observe_lite().shape[-1]

policy = MLPPolicy(obs_dim, hidden=64, rng=0)
print("obs_dim:", obs_dim)
print("genome parameters:", policy.n_params)
print(evaluate_controller(policy, envs=1024))

obs_dim: 13
genome parameters: 1091
{'mean_reward_per_player': array([ 128., -128.], dtype=float32), 'win_rate_per_player': array([1., 0.], dtype=float32), 'mean_length': np.float64(384.0)}


## 4. Genetic algorithm starter

This is intentionally simple:

1. Keep a population of MLP genomes.
2. Evaluate each genome.
3. Keep elites.
4. Refill the population with mutation + crossover.

In [6]:
def evaluate_genome(genome, obs_dim, *, hidden=32, envs=1024, seed=0):
    policy = MLPPolicy(obs_dim, hidden=hidden, genome=genome)
    scores = evaluate_controller(policy, envs=envs, seed=seed)

    # For symmetric self-play, player 0 and player 1 share the same policy.
    # Mean reward is a weak signal; win rate and survival length often help.
    return float(
        scores["mean_reward_per_player"].mean()
        + 0.25 * scores["win_rate_per_player"].mean()
        + 0.001 * scores["mean_length"]
    )


def make_child(parent_a, parent_b, rng, mutation_std=0.03, mutation_rate=0.05):
    mask = rng.random(parent_a.shape) < 0.5
    child = np.where(mask, parent_a, parent_b).copy()

    mutate = rng.random(child.shape) < mutation_rate
    child[mutate] += rng.normal(0, mutation_std, size=mutate.sum())
    return child.astype(np.float32)

# run_ga: run
def run_ga(
    generations=20,
    pop_size=40,
    elite_count=6,
    hidden=32,
    eval_envs=1024,
    seed=0,
):
    rng = np.random.default_rng(seed)

    environment_variables = TronBatchModel(width=WIDTH, height=HEIGHT, players=2, envs=1)
    found_parameters = environment_variables.observe_lite().shape[-1]
    n_params = MLPPolicy(found_parameters, hidden=hidden).n_params

    population = rng.normal(0, 0.2, size=(pop_size, n_params)).astype(np.float32)
    best = None

    for gen in range(generations):
        fitness = np.array([
            evaluate_genome(g, found_parameters, hidden=hidden, envs=eval_envs, seed=seed + gen)
            for g in population
        ])

        order = np.argsort(fitness)[::-1]
        population = population[order]
        fitness = fitness[order]

        if best is None or fitness[0] > best[0]:
            best = (float(fitness[0]), population[0].copy())

        print(f"gen={gen:03d} best={fitness[0]:+.4f} mean={fitness.mean():+.4f}")

        elites = population[:elite_count]
        new_pop = [e.copy() for e in elites]

        while len(new_pop) < pop_size:
            a, b = rng.choice(elite_count, size=2, replace=True)
            new_pop.append(make_child(elites[a], elites[b], rng))

        population = np.stack(new_pop)

    best_fitness, best_genome = best
    return best_fitness, best_genome, found_parameters

genetic_algo_params = {
    "hidden": 16,
    'generations': 8,
    'pop_size': 16,
    'elite_count': 4,
    'eval_envs': 512,
    'seed': 42,
}

best_fitness, best_genome, obs_dim = run_ga(
    **genetic_algo_params
)

print("best fitness:", best_fitness)

gen=000 best=+0.6172 mean=-59.4826
gen=001 best=+0.6811 mean=-11.4177
gen=002 best=+0.6678 mean=-15.6852
gen=003 best=+0.7455 mean=+0.3411
gen=004 best=+0.7457 mean=-7.6281
gen=005 best=+0.7458 mean=-17.4301
gen=006 best=+0.7476 mean=+0.7148
gen=007 best=+0.7484 mean=+0.7218
best fitness: 0.748396484375


## 5. Save and reload a trained genome

This saves the best genome into `./tron_genomes/` using a filename that includes key parameters and the current date/time. It also writes a `.json` metadata sidecar with the training settings.


In [7]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np


def export_genome(
    genome,
    *,
    obs_dim,
    hidden,
    fitness=None,
    generations=None,
    pop_size=None,
    elite_count=None,
    eval_envs=None,
    width=32,
    height=32,
    players=2,
    seed=None,
    out_dir=Path("tron_genomes"),
):
    """
    Save a genome and metadata into ./tron_genomes/.

    Files created:
        tron_genomes/<name>.npy
        tron_genomes/<name>.json
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    name = (
        f"tron_"
        f"p{players}_"
        f"{width}x{height}_"
        f"obs{obs_dim}_"
        f"h{hidden}_"
        f"pop{pop_size}_"
        f"gen{generations}_"
        f"eval{eval_envs}_"
        f"seed{seed}_"
        f"{timestamp}"
    )

    genome_path = out_dir / f"{name}.npy"
    metadata_path = out_dir / f"{name}.json"

    np.save(genome_path, np.asarray(genome, dtype=np.float32))

    metadata = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "genome_file": genome_path.name,
        "fitness": None if fitness is None else float(fitness),
        "model": {
            "policy_type": "MLPPolicy",
            "obs_dim": int(obs_dim),
            "hidden": int(hidden),
            "action_count": 3,
        },
        "environment": {
            "width": int(width),
            "height": int(height),
            "players": int(players),
        },
        "genetic_algorithm": {
            "generations": None if generations is None else int(generations),
            "pop_size": None if pop_size is None else int(pop_size),
            "elite_count": None if elite_count is None else int(elite_count),
            "eval_envs": None if eval_envs is None else int(eval_envs),
            "seed": None if seed is None else int(seed),
        },
    }

    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    print(f"Saved genome:  {genome_path}")
    print(f"Saved metadata: {metadata_path}")

    return genome_path, metadata_path


# Match these to the run_ga(...) call you used above.
ga_params = {
    "elite_count": 8,
    "eval_envs": 512,
    "players": 2,

    **genetic_algo_params,
}

genome_path, metadata_path = export_genome(
    best_genome,
    obs_dim=obs_dim,
    fitness=best_fitness,
    **ga_params,
)

loaded = np.load(genome_path)
trained_policy = MLPPolicy(obs_dim, hidden=ga_params["hidden"], genome=loaded)

print(evaluate_controller(trained_policy, envs=512, seed=999))

Saved genome:  tron_genomes\tron_p2_32x32_obs13_h16_pop16_gen8_eval512_seed42_20260505_131805.npy
Saved metadata: tron_genomes\tron_p2_32x32_obs13_h16_pop16_gen8_eval512_seed42_20260505_131805.json
{'mean_reward_per_player': array([ 0.87109375, -0.87109375], dtype=float32), 'win_rate_per_player': array([1., 1.], dtype=float32), 'mean_length': np.float64(498.3203125)}


## 6. Watch the trained genome in the GUI

The current `play_live()` function creates its own controller, so for visualizing a custom trained policy,
use a tiny local loop like this. Run it only on a machine with tkinter GUI support.

In [8]:
#Uncomment this cell when you want to watch a trained policy.

In [9]:
from tron_view import GameView

def watch_policy(policy, players=2, width=WIDTH, height=HEIGHT, scale=16, fps=20, seed=0):
    model = TronBatchModel(width=width, height=height, players=players, envs=1, keep_owner=True, seed=seed)
    view = GameView(model, scale=scale, fps=fps)
    try:
        while view.poll():
            if view.take_restart_request():
                model.reset()
            if not model.done[0]:
                model.step(policy.actions(model))
            view.render(model)
    finally:
        view.close()

watch_policy(trained_policy)

## 7. RL setup options

The model already has the key pieces needed for RL:

- `obs = env.observe_lite()`
- `actions = policy(obs)`
- `result = env.step(actions)`
- `result.reward`
- `result.done`
- `env.auto_reset_done()`

For Python 3.14, a pure NumPy loop like the GA above is the lowest-friction start.
For deep RL, you can later wrap `TronBatchModel` in a Gymnasium-style environment or connect it to PyTorch/JAX once your preferred stack supports your Python version.